# 🔬 CV Generator — Step-by-Step Prompt & Node Playground

Welcome to the **Step-by-Step Prompt Engineering Playground**. 
This notebook breaks down the agentic CV generator into individual graph node steps. 
You can inspect, adjust, and re-run prompts at each node to see their exact effect on intermediate data and the final CV.

---

## 0. Environment Setup & Initial Pipeline State

In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

from generation.nodes import (
    node_analyzer,
    node_retriever,
    node_drafter,
    node_refiner,
    node_auditor
)
from generation.helpers import load_prompt
from kb_config import get_wiki_dir

# Set environment defaults
os.environ["CONFIG_FILE"] = "config.yaml"

# Target Job Description Input Widgets
sample_jd = """Senior AI / Machine Learning Engineer
Company: TechCorp Solutions
Location: London, UK (Hybrid)

About the Role:
We are looking for a Senior AI Systems Architect to lead our LLM and Agentic AI pipeline initiatives.
Requirements:
- 5+ years experience building Python, LangChain, and LangGraph LLM applications.
- Experience with knowledge graphs, RAG systems, and document processing.
- Track record of deploying production AI workflows with PyTorch and Transformers.
- Strong leadership, stakeholder communication, and technical mentoring.
"""

jd_file_box = widgets.Text(
    value="",
    placeholder="Optional: enter path to JD file (e.g. job-descriptions/target_jd.txt)",
    description="JD File Path:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='480px')
)

jd_load_btn = widgets.Button(
    description="📂 Load File",
    button_style="info",
    layout=widgets.Layout(width='120px')
)

jd_textarea = widgets.Textarea(
    value=sample_jd,
    placeholder="Paste or edit the target Job Description here...",
    description="Job Description:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='90%', height='220px')
)

update_jd_btn = widgets.Button(
    description="✅ Set Active Job Description",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width='260px')
)

jd_status_out = widgets.Output()

# Global State Dictionary
state = {
    "job_description": jd_textarea.value,
    "job_description_raw": jd_textarea.value,
    "iteration_count": 0,
    "max_iterations": 3,
    "strategy_override": None
}

def on_load_file_clicked(b):
    with jd_status_out:
        jd_status_out.clear_output()
        p_str = jd_file_box.value.strip()
        if not p_str:
            print("⚠️ Please enter a file path.")
            return
        p = Path(p_str)
        if not p.is_file():
            print(f"❌ File not found: {p}")
            return
        try:
            content = p.read_text(encoding="utf-8")
            jd_textarea.value = content
            state["job_description"] = content
            state["job_description_raw"] = content
            print(f"✅ Loaded and set active JD from {p} ({len(content)} characters).")
        except Exception as e:
            print(f"❌ Error reading file: {e}")

def on_update_jd_clicked(b):
    with jd_status_out:
        jd_status_out.clear_output()
        content = jd_textarea.value.strip()
        if not content:
            print("❌ Error: Job Description text is empty.")
            return
        state["job_description"] = content
        state["job_description_raw"] = content
        print(f"✅ Active Job Description updated in pipeline state ({len(content)} characters).")

jd_load_btn.on_click(on_load_file_clicked)
update_jd_btn.on_click(on_update_jd_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>📋 Job Description Input</h3>"),
    widgets.HBox([jd_file_box, jd_load_btn]),
    jd_textarea,
    update_jd_btn,
    jd_status_out
]))

print("✅ Pipeline environment initialized. Interactive JD input ready.")

--- 
## 1. Node A: Analyzer

**Objective:** Analyzes the target Job Description to extract keywords, persona, location, company slug, and regional strategy (`emea`, `us_tech`, `executive`).
* **Prompt File:** `src/prompts/cv_gen/analyzer.txt`

In [ ]:
prompt_path_analyzer = Path("src/prompts/cv_gen/analyzer.txt")
current_analyzer_prompt = load_prompt("analyzer.txt")

analyzer_editor = widgets.Textarea(
    value=current_analyzer_prompt,
    description="Prompt Text:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='90%', height='220px')
)

save_analyzer_btn = widgets.Button(description="💾 Save Prompt to Disk", button_style="info", icon="save")
run_analyzer_btn = widgets.Button(description="▶️ Run Node A (Analyzer)", button_style="success", icon="play")
analyzer_output = widgets.Output()

def on_save_analyzer(b):
    prompt_path_analyzer.write_text(analyzer_editor.value, encoding="utf-8")
    print(f"Saved updated analyzer prompt to {prompt_path_analyzer}")

def on_run_analyzer(b):
    with analyzer_output:
        analyzer_output.clear_output()
        # Temporarily apply edited prompt if changed
        prompt_path_analyzer.write_text(analyzer_editor.value, encoding="utf-8")
        print("⏳ Running Node A: Analyzer...")
        res = node_analyzer(state)
        state.update(res)
        print("✅ Node A Completed!")
        print("\nDetected Target Role:", state.get("target_role"))
        print("Detected Organization Slug:", state.get("target_organization_slug"))
        print("Target Region Strategy:", state.get("target_region"))
        print("Extracted Primary Keywords:", state.get("primary_keywords"))
        print("Target Persona Summary:", state.get("target_persona"))

save_analyzer_btn.on_click(on_save_analyzer)
run_analyzer_btn.on_click(on_run_analyzer)

display(widgets.VBox([
    analyzer_editor,
    widgets.HBox([save_analyzer_btn, run_analyzer_btn]),
    analyzer_output
]))

--- 
## 2. Node B: Retriever

**Objective:** Queries the `llm-wiki/` graph, ranks experiences by keyword alignment & semantic score, deduplicates education, retrieves patents/projects/notes, and builds the skill bridging map.
* **Prompt Files:** `src/prompts/cv_gen/retriever_score.txt` & `skill_bridging_map.txt`

In [ ]:
prompt_path_retriever = Path("src/prompts/cv_gen/retriever_score.txt")
current_retriever_prompt = load_prompt("retriever_score.txt")

retriever_editor = widgets.Textarea(
    value=current_retriever_prompt,
    description="Scoring Prompt:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='90%', height='200px')
)

run_retriever_btn = widgets.Button(description="▶️ Run Node B (Retriever)", button_style="success", icon="play")
retriever_output = widgets.Output()

def on_run_retriever(b):
    with retriever_output:
        retriever_output.clear_output()
        prompt_path_retriever.write_text(retriever_editor.value, encoding="utf-8")
        print("⏳ Running Node B: Retriever...")
        res = node_retriever(state)
        state.update(res)
        print("✅ Node B Completed!")
        experiences = state.get("retrieved_experiences", [])
        print(f"Retrieved & Ranked Experiences ({len(experiences)} total):")
        for exp in experiences[:5]:
            role = exp.get("role", "Unknown Role")
            org = exp.get("organization", "Unknown Org")
            score = exp.get("relevance_score", 0)
            print(f"  - [{score} pts] {role} at {org}")
        print("\nEducation Entries:", len(state.get("education_entries", [])))
        print("Projects Retrieved:", len(state.get("retrieved_projects", [])))
        print("Skill Bridging Map:", state.get("skill_bridging_map"))

run_retriever_btn.on_click(on_run_retriever)

display(widgets.VBox([
    retriever_editor,
    run_retriever_btn,
    retriever_output
]))

--- 
## 3. Node C: Drafter

**Objective:** Synthesizes the retrieved Wiki knowledge graph entries, regional strategy guidelines, and persona into a tailored Markdown CV.
* **Prompt Files:** `src/prompts/cv_gen/drafter_system.txt` & `drafter_user.txt`

In [ ]:
prompt_path_drafter_sys = Path("src/prompts/cv_gen/drafter_system.txt")
current_drafter_sys = load_prompt("drafter_system.txt")

drafter_sys_editor = widgets.Textarea(
    value=current_drafter_sys,
    description="System Prompt:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='90%', height='220px')
)

run_drafter_btn = widgets.Button(description="▶️ Run Node C (Drafter)", button_style="success", icon="play")
drafter_output = widgets.Output()

def on_run_drafter(b):
    with drafter_output:
        drafter_output.clear_output()
        prompt_path_drafter_sys.write_text(drafter_sys_editor.value, encoding="utf-8")
        print("⏳ Running Node C: Drafter (LLM generation)...")
        res = node_drafter(state)
        state.update(res)
        draft = state.get("draft_cv", "")
        print(f"✅ Node C Completed! Draft CV generated ({len(draft.splitlines())} lines).")
        print("\n" + "="*60 + "\n📌 GENERATED CV DRAFT:\n" + "="*60)
        display(Markdown(draft))

run_drafter_btn.on_click(on_run_drafter)

display(widgets.VBox([
    drafter_sys_editor,
    run_drafter_btn,
    drafter_output
]))

--- 
## 4. Node D & E: Refiner & Auditor

**Objective:** Performs line density & page count checks (Refiner) and runs ATS audit critique (Auditor) to generate PASS/FAIL feedback.
* **Prompt File:** `src/prompts/cv_gen/auditor.txt`

In [ ]:
prompt_path_auditor = Path("src/prompts/cv_gen/auditor.txt")
current_auditor_prompt = load_prompt("auditor.txt")

auditor_editor = widgets.Textarea(
    value=current_auditor_prompt,
    description="Auditor Prompt:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='90%', height='200px')
)

run_audit_btn = widgets.Button(description="▶️ Run Node D & E (Refiner & Auditor)", button_style="success", icon="play")
audit_output = widgets.Output()

def on_run_audit(b):
    with audit_output:
        audit_output.clear_output()
        prompt_path_auditor.write_text(auditor_editor.value, encoding="utf-8")
        print("⏳ Running Node D: Refiner...")
        res_refiner = node_refiner(state)
        state.update(res_refiner)
        
        print("⏳ Running Node E: Auditor...")
        res_auditor = node_auditor(state)
        state.update(res_auditor)
        
        print("✅ Audit Completed!")
        print("Refiner Feedback:", state.get("refiner_feedback", "None"))
        print("Audit Result:", state.get("audit_feedback", "None"))

run_audit_btn.on_click(on_run_audit)

display(widgets.VBox([
    auditor_editor,
    run_audit_btn,
    audit_output
]))